# Agent Platform 上での PyTorch ベースのエンドツーエンド ML ワークフローの構築

**学習目標:**
1. BigQuery を使用してトレーニング用と検証用の分割データを作成する方法を学ぶ
1. PyTorch モデルを Docker コンテナにラップし、Agent Platform 上でトレーニングする方法を学ぶ
1. Agent Platform のハイパーパラメータ調整エンジンを使用して最適なハイパーパラメータを見つける方法を学ぶ
1. トレーニング済みの PyTorch モデルを REST API として Agent Platform にデプロイし、クエリを実行する方法を学ぶ

このラボでは、マルチクラス分類のトレーニングアプリケーションを開発し、Docker イメージとしてパッケージ化して、Agent Platform 上でアプリケーションを実行します。このラボで使用する[データセット](../../../datasets/covertype/README.md)は、UCI Machine Learning Repository の **Covertype Data Set** に基づいています。

トレーニングコードでは、データの前処理とモデリングに `PyTorch` を使用します。コードには `hypertune` パッケージが組み込まれており、**Agent Platform** のハイパーパラメータ調整で使用できるようになっています。

## セットアップ

In [ ]:
import json
import os
import time
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from google.cloud import aiplatform, bigquery
from google.cloud.aiplatform import hyperparameter_tuning as hpt
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchtext.vocab import vocab

%load_ext google.cloud.bigquery

## 環境設定の構成

ロケーションパス、接続文字列、その他の環境設定を設定します。ラボ環境を反映した設定で `REGION` と `ARTIFACT_STORE` を必ず更新してください。

- `REGION` - Agent Platform Training および Prediction のコンピュートリージョン
- `ARTIFACT_STORE` - 同じリージョンに作成された GCS バケット

In [ ]:
REGION = "us-central1"

PROJECT_ID = !(gcloud config get-value core/project)
PROJECT_ID = PROJECT_ID[0]

ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"

DATA_ROOT = f"{ARTIFACT_STORE}/data"
JOB_DIR_ROOT = f"{ARTIFACT_STORE}/jobs"
TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"
API_ENDPOINT = f"{REGION}-aiplatform.googleapis.com"

In [ ]:
os.environ["JOB_DIR_ROOT"] = JOB_DIR_ROOT
os.environ["TRAINING_FILE_PATH"] = TRAINING_FILE_PATH
os.environ["VALIDATION_FILE_PATH"] = VALIDATION_FILE_PATH
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION

`ARTIFACT_STORE` バケットが存在しない場合は作成します。このバケットは、変数 `REGION` で指定されたリージョンに作成する必要がある点に注意してください（`REGION` とは異なるリージョンにこの名前のバケットがすでに存在する場合は、下のセルのコマンドで `REGION` にバケットを再作成できるように、`ARTIFACT_STORE` の名前を変更してください）。

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

## BigQuery へのデータセットのインポート

In [ ]:
%%bash

DATASET_LOCATION=US
DATASET_ID=covertype_dataset
TABLE_ID=covertype
DATA_SOURCE=gs://asl-public-data/data/covertype/dataset.csv
SCHEMA=Elevation:INTEGER,\
Aspect:INTEGER,\
Slope:INTEGER,\
Horizontal_Distance_To_Hydrology:INTEGER,\
Vertical_Distance_To_Hydrology:INTEGER,\
Horizontal_Distance_To_Roadways:INTEGER,\
Hillshade_9am:INTEGER,\
Hillshade_Noon:INTEGER,\
Hillshade_3pm:INTEGER,\
Horizontal_Distance_To_Fire_Points:INTEGER,\
Wilderness_Area:STRING,\
Soil_Type:STRING,\
Cover_Type:INTEGER

exists=$(bq ls -d | grep -w $DATASET_ID)
if [ -n "$exists" ]; then
   echo "$DATASET_ID already exists"
else
   echo "Creating $dataset"
   bq --location=$DATASET_LOCATION --project_id=$PROJECT_ID mk --dataset $DATASET_ID
fi

bq --project_id=$PROJECT_ID --dataset_id=$DATASET_ID load \
--source_format=CSV \
--skip_leading_rows=1 \
--replace \
$TABLE_ID \
$DATA_SOURCE \
$SCHEMA

## Covertype データセットの探索

In [ ]:
%%bigquery
SELECT *
FROM `covertype_dataset.covertype`

## トレーニング用および検証用分割データの作成

BigQuery を使用してトレーニング用および検証用の分割データをサンプリングし、Cloud Storage（GCS）に保存します。
### トレーニング用分割データの作成

In [ ]:
!bq query \
-n 0 \
--destination_table covertype_dataset.training \
--replace \
--use_legacy_sql=false \
'SELECT * \
FROM `covertype_dataset.covertype` AS cover \
WHERE \
MOD(ABS(FARM_FINGERPRINT(TO_JSON_STRING(cover))), 10) IN (1, 2, 3, 4)' 

In [ ]:
!bq extract \
--destination_format CSV \
covertype_dataset.training \
$TRAINING_FILE_PATH

### 検証用分割データの作成

In [ ]:
!bq query \
-n 0 \
--destination_table covertype_dataset.validation \
--replace \
--use_legacy_sql=false \
'SELECT * \
FROM `covertype_dataset.covertype` AS cover \
WHERE \
MOD(ABS(FARM_FINGERPRINT(TO_JSON_STRING(cover))), 10) IN (8)' 

In [ ]:
!bq extract \
--destination_format CSV \
covertype_dataset.validation \
$VALIDATION_FILE_PATH

In [ ]:
df_train = pd.read_csv(TRAINING_FILE_PATH)
df_validation = pd.read_csv(VALIDATION_FILE_PATH)

## トレーニングアプリケーションの開発
それでは、前処理とトレーニングアプリケーションを定義しましょう。

### データの前処理
このデータには、数値特徴量とカテゴリ特徴量の両方が含まれています。数値特徴量は、身長や体重など、測定またはカウントできる値です。カテゴリ特徴量は、性別や色など、離散的な値を持ちます。

分析に向けてデータを準備するには、数値特徴量を標準化し、カテゴリ特徴量をワンホットエンコードする必要があります。数値特徴量の標準化とは、平均が 0、標準偏差が 1 になるように値を調整することを意味します。カテゴリ特徴量のワンホットエンコーディングとは、各カテゴリに対して個別の列を作成し、その特徴量がどのカテゴリに属するかを示すために 1 または 0 の値を割り当てることを意味します。

これらの前処理手順を実行するために、数値特徴量とカテゴリ特徴量を別々のリストに配置し、それぞれのリストに適切なロジックを適用します。

In [ ]:
CATEGORICAL_FEATURES = ["Wilderness_Area", "Soil_Type"]
NUMERICAL_FEATURES = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]
LABEL_COLUMN = "Cover_Type"

それでは、実際の前処理ロジックを定義します。

数値特徴量を正規化し、カテゴリ特徴量のワンホットエンコード版を作成します。どちらの特徴量変換にも 2 つのステップがあります。
1. fit ステップ: データセット全体を処理し、変換に必要な「状態」または統計量を計算します（例: 数値特徴量の正規化のための平均と標準偏差、カテゴリ特徴量のワンホットエンコーディングのための値と整数のマッピング）。
2. transform ステップ: 特徴量の変換を実際に適用します。

そのため、`fit()`、`transform()`、`serialize_constants` メソッドを持つ前処理用の `OneHotEncoder` クラスと `StandardScaler` クラスをそれぞれ定義します。

In [ ]:
class OneHotEncoder:
    def fit(self, series: pd.Series):
        categories = series.unique().tolist()
        dictionary = vocab(OrderedDict([(cat, 1) for cat in categories]))
        self.dictionary = dictionary
        return self

    def transform(self, feature: list, dictionary=None) -> torch.Tensor:
        if dictionary:
            assert type(dictionary) == list
            self.dictionary = vocab(
                OrderedDict([(cat, 1) for cat in dictionary])
            )

        indices = self.dictionary.lookup_indices(feature)
        indices = torch.tensor(indices, dtype=torch.long)
        one_hot = F.one_hot(
            indices, num_classes=len(self.dictionary.get_itos())
        )
        return torch.tensor(one_hot).float()

    def serialize_constants(self):
        return {"dictionary": self.dictionary.get_itos()}


class StandardScaler:
    def fit(self, series: pd.Series):
        self.mean = np.float64(series.mean())
        self.std = np.float64(series.std())
        return self

    def transform(self, feature: list, mean=None, std=None) -> torch.Tensor:
        if mean:
            self.mean = np.float64(mean)
        if std:
            self.std = np.float64(std)

        standardized = (feature - self.mean) / self.std
        return torch.tensor(standardized)[:, None].float()

    def serialize_constants(self):
        return {"mean": self.mean, "std": self.std}

これは、`transform` 関数を呼び出してすべての特徴量を変換し、すべての列を連結するユーティリティ関数です。

In [ ]:
def preprocess(df, transformers):
    transformed_features = [
        transformers[c].transform(df[c].to_list())
        for c in df.columns
        if c != LABEL_COLUMN
    ]
    features = torch.cat(transformed_features, 1)

    label = df[LABEL_COLUMN].to_list()
    label = torch.LongTensor(label)
    return features, label

それでは、すべての列に対して個別に `fit()` 関数を呼び出し、`preprocess` 関数で変換しましょう。

In [ ]:
# カテゴリ列の前処理
transformers = {
    c_feature: OneHotEncoder().fit(df_train[c_feature])
    for c_feature in CATEGORICAL_FEATURES
}
transformers.update(
    {
        n_feature: StandardScaler().fit(df_train[n_feature])
        for n_feature in NUMERICAL_FEATURES
    }
)

training_transformed = preprocess(df_train, transformers)
validation_transformed = preprocess(df_validation, transformers)

### 予測用の前処理状態ファイルのエクスポート
トレーニングデータと検証データの変換が正常に完了したら、`serialize_constants` を使用して状態を JSON ファイルに保存できます。

In [ ]:
# 前処理用の JSON をエクスポート
preprocessing_json = {
    c: transformers[c].serialize_constants()
    for c in df_train.columns
    if c != LABEL_COLUMN
}

with open("preprocessing.json", "w") as f:
    json.dump(preprocessing_json, f)

In [ ]:
preprocessing_json

### モデルおよびトレーニング/検証ステップの定義
PyTorch でトレーニングおよび検証ステップを含むシンプルなニューラルネットワークモデルを定義します。

PyTorch では、`torch.nn.Module` をサブクラス化するクラスでニューラルネットワークモデルを定義できます。<br>
モデルのレイヤーやその他のコンポーネントを定義する `__init__()` メソッドと、計算が実行される `forward()` メソッドがあります。<br>
詳細については、[公式ドキュメント](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module)を参照してください。

また、CPU または GPU デバイスのいずれでもモデルを実行できるように、CUDA の可用性に応じて [`torch.device`](https://pytorch.org/docs/stable/tensor_attributes.html#torch-device) を定義し、パラメータのデータ型変換を処理する `torch.nn.Module.to()` メソッドで指定します。

In [ ]:
# デバイスの設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device is {device}")


# モデル
class NeuralNetworkModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(54, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 7),
        )

    def forward(self, x):
        return self.linear_relu_stack(x)


model = NeuralNetworkModel().to(device)

[クロスエントロピー損失（cross entropy loss）](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) と [Adam オプティマイザ](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) を使用します。

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

以下の操作を含むトレーニングステップを定義しましょう。
- `Tensor.to()` を呼び出して、テンソルを指定したデバイスに割り当てる。
- 勾配をゼロにリセットする（`optimizer.zero_grad()`）。
- モデルの予測を実行する。上記で定義した `Module.forward()` 関数を呼び出す代わりに、単に `Module()` を呼び出すことができます。これは `forward()` だけでなく、登録された他のフックの実行も処理するためです。
- 損失を計算し（`loss_fn(pred, y)`）、パラメータに逆伝播する（`loss.backward()`）。
- オプティマイザを更新して次のステップに備える（`optimizer.step()`）。

In [ ]:
def train(dataloader, model, loss_fn, optimizer, device):
    model.train()
    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

同様の方法で、検証ステップを定義します。<br>
ただし、以下の追加の操作に注意してください。
- モデルを評価モードに切り替えるための `model.eval()`。これにより、一部の操作（Dropout、Batch Normalization など）の動作が自動的に切り替わります。
- 勾配計算を無効にするための `with torch.no_grad()`。これにより、メモリ消費量を削減できます。
- 検証損失とともに精度（accuracy）を計算して返す。

In [ ]:
def validation(dataloader, model, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    validation_loss = 0
    correct = 0

    model.eval()
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            validation_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    validation_loss /= num_batches
    accuracy = correct / size
    return accuracy, validation_loss

### ローカルでのトレーニングの実行
Agent Platform にトレーニングを渡す前に、ローカルで実行できるかをテストしましょう。

これにより、コード内のエラーをローカルで特定して修正するのに役立ちます。

In [ ]:
# Dataset と DataLoader
BATCH_SIZE = 16

train_dataset = TensorDataset(*training_transformed)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
validation_dataset = TensorDataset(*validation_transformed)
validation_dataloader = DataLoader(validation_dataset, batch_size=BATCH_SIZE)

In [ ]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}")
    train(train_dataloader, model, loss_fn, optimizer, device)
    accuracy, validation_loss = validation(
        validation_dataloader, model, loss_fn, device
    )
    print(
        ", ".join(
            (
                f"Validation Accuracy: {(100*accuracy):>0.1f}%",
                f"Avg loss: {validation_loss:>8f}",
            )
        )
    )
print("Done!")

## ハイパーパラメータ調整
### ハイパーパラメータ調整アプリケーションの準備
このデータセットでのトレーニングの実行は計算コストが高いため、Agent Platform Training で分散ハイパーパラメータ調整ジョブを実行する利点があります。

In [ ]:
TRAINING_APP_FOLDER = "training_app"
os.makedirs(TRAINING_APP_FOLDER, exist_ok=True)

### チューニングスクリプトの作成
Agent Platform でトレーニングを実行するために、前処理からトレーニングまでのすべてのコードを含む Python ファイルを定義します。<br>
コードの大部分は上記のセルにあるものと同じです。ただし、ハイパーパラメータ調整を実行するために、コードを数行追加します。これらの行は、後で使用できるようにトレーニング済みモデルと前処理の状態を保存します。

最適化の指標として `accuracy`（精度）を報告するために、`hypertune` ライブラリを使用します。

In [ ]:
%%writefile {TRAINING_APP_FOLDER}/train.py
import os
import subprocess
import sys
import json

import fire
import hypertune
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchtext.vocab import vocab
from collections import OrderedDict
import torch.nn.functional as F


CATEGORICAL_FEATURES = ["Wilderness_Area", "Soil_Type"]
NUMERICAL_FEATURES = ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology',
       'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways',
       'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm',
       'Horizontal_Distance_To_Fire_Points']
LABEL_COLUMN = "Cover_Type"


class OneHotEncoder:
    def fit(self, series: pd.Series):
        categories = series.unique().tolist()
        dictionary = vocab(OrderedDict([(cat, 1) for cat in categories]))
        self.dictionary = dictionary
        return self

    def transform(self, feature:list, dictionary=None) -> torch.Tensor:
        if dictionary:
            assert type(dictionary) == list
            self.dictionary = vocab(
                OrderedDict([(cat, 1) for cat in dictionary])
            )

        indices = self.dictionary.lookup_indices(feature)
        indices = torch.tensor(indices, dtype=torch.long)
        one_hot = F.one_hot(
            indices, num_classes=len(self.dictionary.get_itos())
        )
        return torch.tensor(one_hot).float()

    def serialize_constants(self):
        return {"dictionary": self.dictionary.get_itos()}


class StandardScaler:
    def fit(self, series: pd.Series):
        self.mean = np.float64(series.mean())
        self.std = np.float64(series.std())
        return self

    def transform(self, feature:list, mean=None, std=None) -> torch.Tensor:
        if mean:
            self.mean = np.float64(mean)
        if std:
            self.std = np.float64(std)

        standardized = (feature - self.mean) / self.std
        return torch.tensor(standardized)[:, None].float()

    def serialize_constants(self):
        return {"mean": self.mean, "std": self.std}

def preprocess(df, transformers):
    transformed_features = [transformers[c].transform(df[c].to_list()) 
                            for c in df.columns if c != LABEL_COLUMN]
    features = torch.cat(transformed_features, 1)
    
    label = df[LABEL_COLUMN].to_list()
    label = torch.LongTensor(label)
    return features, label

    
def train_evaluate(job_dir, training_dataset_path, validation_dataset_path, batch_size, max_iter, hptune):
    
    df_train = pd.read_csv(training_dataset_path)
    df_validation = pd.read_csv(validation_dataset_path)

    if not hptune:
        df_train = pd.concat([df_train, df_validation])

    
    # カテゴリ列の前処理
    transformers = {c_feature: OneHotEncoder().fit(df_train[c_feature]) 
                    for c_feature in CATEGORICAL_FEATURES}
    transformers.update({n_feature: StandardScaler().fit(df_train[n_feature]) 
                         for n_feature in NUMERICAL_FEATURES})

    training_transformed = preprocess(df_train, transformers)
    validation_transformed = preprocess(df_validation, transformers)

    # Dataset と DataLoader
    train_dataset = TensorDataset(*training_transformed)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)
    validation_dataset = TensorDataset(*validation_transformed)
    validation_dataloader = DataLoader(validation_dataset, batch_size=batch_size)
    
    # デバイスの設定
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # モデル
    class NeuralNetworkModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.linear_relu_stack = nn.Sequential(
                nn.Linear(54, 32),
                nn.ReLU(),
                nn.Linear(32, 16),
                nn.ReLU(),
                nn.Linear(16, 7)
            )

        def forward(self, x):
            return self.linear_relu_stack(x)

    model = NeuralNetworkModel().to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    def train(dataloader, model, loss_fn, optimizer, device):
        model.train()
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
    
    def validation(dataloader, model, loss_fn, device):
        size = len(dataloader.dataset)
        num_batches = len(dataloader)
        validation_loss = 0
        correct = 0

        model.eval()
        with torch.no_grad():
            for X, y in dataloader:
                X = X.to(device)
                y = y.to(device)
                pred = model(X)
                validation_loss += loss_fn(pred, y).item()
                correct += (pred.argmax(1) == y).type(torch.float).sum().item()
        validation_loss /= num_batches
        correct /= size
        return correct, validation_loss

    epochs = max_iter
    for t in range(epochs):
        print(f"Epoch {t+1}")
        train(train_dataloader, model, loss_fn, optimizer, device)

    if hptune:
        accuracy, _ = validation(validation_dataloader, model, loss_fn, device)
        # hypertune でログに記録
        hpt = hypertune.HyperTune()
        hpt.report_hyperparameter_tuning_metric(
          hyperparameter_metric_tag='accuracy',
          metric_value=accuracy
        )

    if not hptune:
        # モデルを保存
        model_filename = "model.pt"
        model_scripted = torch.jit.script(model)
        model_scripted.save(model_filename)
        gcs_model_path = "{}/{}".format(job_dir, model_filename)

        # 前処理用の JSON をエクスポート
        preprocesinng_json = {c: transformers[c].serialize_constants() 
                              for c in df_train.columns if c != LABEL_COLUMN}
        preproc_json_filename = 'preprocessing.json'
        with open(preproc_json_filename, 'w') as f:
            json.dump(preprocesinng_json, f)
        gcs_preprocessing_json_path = "{}/{}".format(job_dir, preproc_json_filename)

        # ファイルを GCS に送信
        subprocess.check_call(['gcloud', 'storage', 'cp', model_filename, gcs_model_path], stderr=sys.stdout)
        subprocess.check_call(['gcloud', 'storage', 'cp', preproc_json_filename, gcs_preprocessing_json_path], 
                              stderr=sys.stdout)


if __name__ == "__main__":
    fire.Fire(train_evaluate)

### スクリプトを Docker イメージにパッケージ化する
コンテナベースのクラウドトレーニングを実行するために、Dockerfile を定義します。<br>
Artifact Registry から事前定義された PyTorch イメージを使用し、`cloud-hypertune` や `torchtext` を含む追加のライブラリをトレーニングイメージにインストールしていることに注目してください。

In [ ]:
%%writefile {TRAINING_APP_FOLDER}/Dockerfile

FROM us-docker.pkg.dev/vertex-ai/training/pytorch-xla.2-3.py310:latest
RUN pip install -U torchtext==0.18.0 fire
WORKDIR /app
COPY train.py .

ENTRYPOINT ["python", "train.py"]

### Docker イメージのビルド

ベースイメージの URI がプロジェクトの **Artifact Registry** を指すように更新されていることを確認してください。<br>
**Cloud Build** を使用してイメージをビルドし、プロジェクトの **Artifact Registry** にプッシュします。リモートクラウドサービスを使用してイメージをビルドするため、Docker をローカルにインストールする必要はありません。

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "pytorch_trainer_image"
IMAGE_TAG = "latest"
IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{IMAGE_TAG}"

os.environ["IMAGE_URI"] = IMAGE_URI

In [ ]:
!gcloud builds submit --tag $IMAGE_URI $TRAINING_APP_FOLDER

### Agent Platform ハイパーパラメータ調整ジョブの送信

また、ハイパーパラメータ調整ジョブの仕様を定義する YAML ファイルを作成しましょう。<br>
トレーニングアプリケーションは、モデルを制御する 2 つのハイパーパラメータを受け入れるように設計されています。
- 最大イテレーション数（Max Iterations）
- バッチサイズ（Batch Size）

以下のファイルは、Agent Platform のハイパーパラメータ調整を構成して、最大 5 つのトライアルを並行して実行し、`max_iter` には 2 つの離散値から、`batch_size` には `16` から `128` までの線形範囲から選択するようにします。

In [ ]:
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"forestcover_tuning_{TIMESTAMP}"
JOB_DIR = f"{JOB_DIR_ROOT}/{JOB_NAME}"

args = [
    f"--job_dir={JOB_DIR}",
    f"--training_dataset_path={TRAINING_FILE_PATH}",
    f"--validation_dataset_path={VALIDATION_FILE_PATH}",
    f"--hptune",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": IMAGE_URI,
            "args": args,
        },
    }
]

custom_job = aiplatform.CustomJob(
    display_name=JOB_NAME,
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{JOB_DIR}/staging",
)

In [ ]:
hpt_job = aiplatform.HyperparameterTuningJob(
    display_name=JOB_NAME,
    custom_job=custom_job,
    metric_spec={
        "accuracy": "maximize",
    },
    parameter_spec={
        "batch_size": hpt.IntegerParameterSpec(min=16, max=128, scale="linear"),
        "max_iter": hpt.DiscreteParameterSpec(values=[10, 20], scale="linear"),
    },
    max_trial_count=5,
    parallel_trial_count=5,
)

hpt_job.run()

**注**: 参考として、YAML ファイルで構成を指定できる同等の gcloud ai コマンドを以下に示します。

```bash
MACHINE_TYPE="n1-standard-4"
REPLICA_COUNT=1
CONFIG_YAML=config.yaml

cat <<EOF > $CONFIG_YAML
studySpec:
  metrics:
  - metricId: accuracy
    goal: MAXIMIZE
  parameters:
  - parameterId: max_iter
    discreteValueSpec:
      values:
      - 10
      - 20
  - parameterId: batch_size
    integerValueSpec:
      minValue: 16
      maxValue: 128
    scaleType: UNIT_LINEAR_SCALE
  algorithm: ALGORITHM_UNSPECIFIED # results in Bayesian optimization
trialJobSpec:
  workerPoolSpecs:  
  - machineSpec:
      machineType: $MACHINE_TYPE
    replicaCount: $REPLICA_COUNT
    containerSpec:
      imageUri: $IMAGE_URI
      args:
      - --job_dir=$JOB_DIR
      - --training_dataset_path=$TRAINING_FILE_PATH
      - --validation_dataset_path=$VALIDATION_FILE_PATH
      - --hptune
EOF

gcloud ai hp-tuning-jobs create     --region=$REGION     --display-name=$JOB_NAME     --config=$CONFIG_YAML     --max-trial-count=5     --parallel-trial-count=5

echo "JOB_NAME: $JOB_NAME"
```

Agent Platform Training ダッシュボードに移動し、[Hyperparameter Tuning Jobs] でハイパーパラメータ調整ジョブの進行状況を確認します。

### ハイパーパラメータ調整結果の取得

ジョブが完了したら、GCP コンソールを使用するか、以下の関数を使用してプログラムで結果を確認できます（このコードは、ハイパーパラメータ調整エンジンが最適化する指標が最大化されることを前提としている点に注意してください）。

In [ ]:
def get_best_hyperparameters(hpt_job):
    metrics = [
        trial.final_measurement.metrics[0].value for trial in hpt_job.trials
    ]
    best_trial = hpt_job.trials[metrics.index(max(metrics))]
    best_parameters = {
        param.parameter_id: param.value for param in best_trial.parameters
    }
    return int(best_parameters["batch_size"]), int(best_parameters["max_iter"])


batch_size, max_iter = get_best_hyperparameters(hpt_job)
print(f"best batch_size: {batch_size}, best max_iter: {max_iter}")

## 最適なハイパーパラメータでモデルを再トレーニングする

これで、最適なハイパーパラメータを使用し、トレーニング用と検証用の分割データを組み合わせたものをトレーニングデータセットとして使用して、モデルを再トレーニングできるようになりました。

### トレーニングジョブの構成と実行

In [ ]:
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"forestcover_training_{TIMESTAMP}"
JOB_DIR = f"{JOB_DIR_ROOT}/{JOB_NAME}"

args = [
    f"--job_dir={JOB_DIR}",
    f"--training_dataset_path={TRAINING_FILE_PATH}",
    f"--validation_dataset_path={VALIDATION_FILE_PATH}",
    f"--batch_size={batch_size}",
    f"--max_iter={max_iter}",
    f"--nohptune",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": IMAGE_URI,
            "args": args,
        },
    }
]

custom_job = aiplatform.CustomJob(
    display_name=JOB_NAME,
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{JOB_DIR}/staging",
)

custom_job.run()

**注**: 参考として、コンテナトレーニングを実行する同等の bash コマンドを以下に示します。

Python セルで変数を設定します。

```python
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"JOB_{TIMESTAMP}"
JOB_DIR = f"{JOB_DIR_ROOT}/{JOB_NAME}"

MACHINE_TYPE="n1-standard-4"
REPLICA_COUNT=1

WORKER_POOL_SPEC = f"""\
machine-type={MACHINE_TYPE},\
replica-count={REPLICA_COUNT},\
container-image-uri={IMAGE_URI}\
"""

ARGS = f"""\
--job_dir={JOB_DIR},\
--training_dataset_path={TRAINING_FILE_PATH},\
--validation_dataset_path={VALIDATION_FILE_PATH},\
--batch_size={batch_size},\
--max_iter={max_iter},\
--nohptune\
"""
```

以下の bash コマンドを実行します。

```bash
!gcloud ai custom-jobs create   --region={REGION}   --display-name={JOB_NAME}   --worker-pool-spec={WORKER_POOL_SPEC}   --args={ARGS}
```

### トレーニング出力の確認

トレーニングスクリプトは、トレーニング済みモデルを GCS 上の `JOB_DIR` フォルダに 'model.pkl' として保存しました。

**注:** 以下のセルを実行する前に、上記のセルによってトリガーされたジョブが完了するのを待つ必要があります。

In [ ]:
!gcloud storage ls $JOB_DIR

## Agent Platform Prediction へのモデルのデプロイ
モデルのトレーニングが完了しました。次に、Agent Platform Prediction を使用してモデルをデプロイします。

In [ ]:
PREDICT_APP_FOLDER = "predict_app"
os.makedirs(PREDICT_APP_FOLDER, exist_ok=True)
os.environ["PREDICT_APP_FOLDER"] = PREDICT_APP_FOLDER
os.environ["JOB_DIR"] = JOB_DIR

### TorchServe ハンドラーの作成
予測ハンドラーには TorchServe を使用します。カスタムの前処理ステップがあるため、`BaseHandler` をサブクラス化してカスタムハンドラーを定義します。<br>

任意の名前にしたクラスを作成することでカスタムハンドラーを作成できますが、`initialize()` メソッドと `handle()` メソッドを持つ必要があります。
- `initialize()`: このメソッドはモデルの読み込み時に呼び出されます。この関数に状態ファイルを読み込むロジックを追加します。
- `handle()`: このクラスのメイン関数です。この関数内にステップを定義します。

多くの場合、`preprocess()`、`inference()`、`postprocess()` 関数をオーバーライドして、それぞれカスタムの前処理、推論、後処理を定義することになります。<br>
また、この Python ファイルでは複数のクラスに対応できないため、前処理クラスから `transform()` 関数をコピーし、`_one_hot_transformer` および `_standard_scale_transformer` として定義します。

詳細については、[TorchServe 公式ドキュメント](https://pytorch.org/serve/custom_service.html) を参照してください。

In [ ]:
%%writefile {PREDICT_APP_FOLDER}/handler.py
from ts.torch_handler.base_handler import BaseHandler

import logging
import os
import json
from collections import OrderedDict

logging.basicConfig(level=logging.INFO)

import numpy as np

import torch
from torchtext.vocab import vocab
import torch.nn.functional as F
    
CATEGORICAL_FEATURES = ["Wilderness_Area", "Soil_Type"]
NUMERICAL_FEATURES = ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology',
       'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways',
       'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm',
       'Horizontal_Distance_To_Fire_Points']

    
class ModelHandler(BaseHandler):
    
    def __init__(self):
        self._context = None
        self.initialized = False
        self.explain = False
        self.target = 0


    def _one_hot_transformer(self, feature, dictionary):
        dictionary = vocab(OrderedDict([(cat, 1) for cat in dictionary]))
        
        indices = dictionary.lookup_indices(feature)
        indices = torch.tensor(indices, dtype=torch.long)
        one_hot = F.one_hot(indices, num_classes=len(dictionary.get_itos()))
        return torch.tensor(one_hot).float()

    def _standard_scale_transformer(self, feature, mean, std):
        standardized = (feature - np.float64(mean)) / np.float64(std)
        return torch.tensor(standardized)[:, None].float()    


    def initialize(self, context):
        self._context = context
        self.initialized = True
        
        # モデルの読み込み
        self.manifest = context.manifest

        properties = context.system_properties
        model_dir = properties.get("model_dir")

        self.device = "cpu"

        # シリアライズされたモデル/pt ファイルの読み込み
        serialized_file = self.manifest['model']['serializedFile']
        model_pt_path = os.path.join(model_dir, serialized_file)
        if not os.path.isfile(model_pt_path):
            raise RuntimeError("Missing the model.pth file")

        self.model = torch.jit.load(model_pt_path)
        
        # 前処理設定の読み込み
        preprocessing_json_path = os.path.join(model_dir, "preprocessing.json")
        with open(preprocessing_json_path) as f:
            self.preprocessing_json = json.load(f)
        
        self.initialized = True


    def preprocess(self, _data):
        data = _data[0]
        
        preprocessed_data = [self._one_hot_transformer(data[c_feature], 
                                                       self.preprocessing_json[c_feature]["dictionary"]) 
                             for c_feature in CATEGORICAL_FEATURES]
        preprocessed_data.extend([self._standard_scale_transformer(data[n_feature],
                                                                   self.preprocessing_json[n_feature]["mean"],
                                                                   self.preprocessing_json[n_feature]["std"]) 
                                  for n_feature in NUMERICAL_FEATURES])
        preprocessed_data = torch.cat(preprocessed_data, 1)
        
        return preprocessed_data

    def inference(self, model_input):
        model_output = self.model.forward(model_input)
        return model_output

    def postprocess(self, inference_output):
        postprocess_output = torch.argmax(inference_output, dim=1)        
        return [postprocess_output.cpu().numpy().tolist()]

    def handle(self, data, context):
        model_input = self.preprocess(data)
        model_output = self.inference(model_input)
        output = self.postprocess(model_output)
        return output

### ファイルのパッケージ化と予測用 .mar ファイルのビルド

クラウドトレーニング中に作成されたトレーニング済みモデルと前処理状態の JSON ファイルをコピーしましょう。

In [ ]:
!gcloud storage cp $JOB_DIR/model.pt ./$PREDICT_APP_FOLDER
!gcloud storage cp $JOB_DIR/preprocessing.json ./$PREDICT_APP_FOLDER

`torch-model-archiver` コマンドを使用して、モデルファイルと前処理状態の JSON ファイルを指定し、予測パッケージを作成します。

詳細については、[公式の README.md](https://github.com/pytorch/serve/blob/master/model-archiver/README.md#torch-model-archiver-for-torchserve) を参照してください。

In [ ]:
%%bash 
torch-model-archiver -f \
  --model-name model \
  --version 1.0 \
  --serialized-file ./$PREDICT_APP_FOLDER/model.pt \
  --handler ./$PREDICT_APP_FOLDER/handler.py \
  --extra-files ./$PREDICT_APP_FOLDER/preprocessing.json \
  --export-path=./$PREDICT_APP_FOLDER

### Agent Platform Prediction へのモデルのデプロイ
次に、モデルを Agent Platform Prediction にデプロイします。

In [ ]:
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
DEPLOY_MODEL_GCS_URI = f"{ARTIFACT_STORE}/pytorch-model-deploy/{TIMESTAMP}"

!gcloud storage cp -r $PREDICT_APP_FOLDER $DEPLOY_MODEL_GCS_URI

In [ ]:
!gcloud storage ls -a -l $DEPLOY_MODEL_GCS_URI

### トレーニング済みモデルのアップロード

In [ ]:
model_display_name = f"pytorch-model-{TIMESTAMP}"
model_description = (
    "PyTorch based forest cover classifier with the pre-built PyTorch image"
)
serving_container_image_uri = (
    "us-docker.pkg.dev/vertex-ai/prediction/pytorch-cpu.2-3:latest"
)

model = aiplatform.Model.upload(
    display_name=model_display_name,
    description=model_description,
    serving_container_image_uri=serving_container_image_uri,
    artifact_uri=DEPLOY_MODEL_GCS_URI,
)

model.wait()

print(model.display_name)
print(model.resource_name)

### アップロードされたモデルのデプロイ

In [ ]:
machine_type = "n1-standard-4"

endpoint = model.deploy(
    machine_type=machine_type, accelerator_type=None, accelerator_count=None
)

### 予測の提供
#### JSON 形式のインスタンスを含む入力ファイルを準備する

In [ ]:
instances = [
    {
        "Elevation": [2841.0],
        "Aspect": [45.0],
        "Slope": [0.0],
        "Horizontal_Distance_To_Hydrology": [644.0],
        "Vertical_Distance_To_Hydrology": [282.0],
        "Horizontal_Distance_To_Roadways": [1376.0],
        "Hillshade_9am": [218.0],
        "Hillshade_Noon": [237.0],
        "Hillshade_3pm": [156.0],
        "Horizontal_Distance_To_Fire_Points": [1003.0],
        "Wilderness_Area": ["Commanche"],
        "Soil_Type": ["C4758"],
    }
]

endpoint.predict(instances)

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.